In [89]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [90]:
#!pip install torchmetrics

In [91]:
#!unzip /content/drive/MyDrive/SESM/PySESM.zip
!mkdir results_1
!mkdir results_2
!mkdir results_3
!mkdir results_1/plots
!mkdir results_2/plots
!mkdir results_3/plots
!mkdir results_1/stats
!mkdir results_2/stats
!mkdir results_3/stats


mkdir: cannot create directory ‘results_1’: File exists
mkdir: cannot create directory ‘results_2’: File exists
mkdir: cannot create directory ‘results_3’: File exists
mkdir: cannot create directory ‘results_1/plots’: File exists
mkdir: cannot create directory ‘results_2/plots’: File exists
mkdir: cannot create directory ‘results_3/plots’: File exists
mkdir: cannot create directory ‘results_1/stats’: File exists
mkdir: cannot create directory ‘results_2/stats’: File exists
mkdir: cannot create directory ‘results_3/stats’: File exists


In [92]:
#!pip install wandb -qU

In [93]:
import torch
import sys

# Agregar la ruta al sys.path
sys.path.append('/content/drive/MyDrive/SESM')

#import wandb
#from torchmetrics import MeanSquaredError
from sklearn.metrics import mean_squared_error

import numpy as np
from scipy.stats import multivariate_normal

from PySESM.models.SESM.SESM import SESM_Model
from PySESM.base_functions.Function import GaussianFunctions
from PySESM.models.ISTALayer import ISTALayer

import pandas as pd

from google.colab import files
import plotly.express as px
import matplotlib.pyplot as plt
import plotly.graph_objects as go

## Login de Wandb

In [94]:
#wandb.login()

## Definicion de covarianzas no diagnonales

In [95]:

# Non-diagonal covariance
def generate_sigma_tensors():
    """
    Generates non-diagonal covariance tensors for 2D Gaussian distributions.

    Returns:
    tuple: Tuple containing three non-diagonal covariance tensors (sigma1, sigma2, sigma3).
    """
    e0 = torch.tensor([1.0, 1.0], dtype=torch.float32)
    e0 = e0 / e0.norm()

    def generate_sigma(rotation_angle, scaling_factors):
        rotation_matrix = torch.tensor([[np.cos(rotation_angle), -np.sin(rotation_angle)],
                                       [np.sin(rotation_angle), np.cos(rotation_angle)]], dtype=torch.float32)
        e = torch.mm(rotation_matrix, e0.unsqueeze(1))
        E = torch.cat((e0.unsqueeze(1), e), dim=1)
        D = torch.diag(torch.tensor(scaling_factors, dtype=torch.float32))
        return torch.mm(torch.mm(E, D), E.t())

    sigma1 = generate_sigma(np.pi/4, [0.4, 0.1])
    sigma2 = generate_sigma(np.pi/4, [0.05, 0.5])
    sigma3 = generate_sigma(np.pi/4, [0.2, 0.5])

    return sigma1, sigma2, sigma3


In [96]:
sigma1, sigma2, sigma3 = generate_sigma_tensors()

## Definicion de varianzas diagonales

In [97]:
sigma1_d = 0.15 * torch.eye(2)
sigma2_d = 0.2 * torch.eye(2)
sigma3_d = 0.3 *torch.eye(2)

## Crear Gaussianas

In [98]:

def generate_mesh_and_z(sigma1, sigma2, sigma3):
    """
    Generates a mesh grid and corresponding probability density values for a mixture
    of three 2D Gaussian distributions.

    Args:
    sigma1 (torch.Tensor): The covariance tensor for the first Gaussian distribution.
    sigma2 (torch.Tensor): The covariance tensor for the second Gaussian distribution.
    sigma3 (torch.Tensor): The covariance tensor for the third Gaussian distribution.

    Returns:
    tuple: Tuple containing mesh grid arrays (xx, yy) and the corresponding probability
    density values (zz).
    """
    N_points = 50
    xl = -2
    xr = 2

    x = np.linspace(xl, xr, N_points+1)
    xx, yy = np.meshgrid(x, x)

    X = torch.tensor(np.column_stack([xx.ravel(), yy.ravel()]), dtype=torch.float32)

    mu1 = torch.tensor([1, 1], dtype=torch.float32)
    mu2 = torch.tensor([1, -1], dtype=torch.float32)
    mu3 = torch.tensor([-1, -1], dtype=torch.float32)

    z1 = torch.tensor(multivariate_normal.pdf(X.numpy(), mu1.numpy(), sigma1.numpy()), dtype=torch.float32)
    z2 = torch.tensor(multivariate_normal.pdf(X.numpy(), mu2.numpy(), sigma2.numpy()), dtype=torch.float32)
    z3 = torch.tensor(multivariate_normal.pdf(X.numpy(), mu3.numpy(), sigma3.numpy()), dtype=torch.float32)

    zz = (z1 + z2 + z3).reshape(xx.shape)

    return xx, yy, zz


## Covarianza de gaussianas del experimento
  - 3 Diagonales
  - 2 Diagonales, 1 no diagonal
  - 1 diagonal, 2 no diagonales
  - 3 no diagonales

In [99]:
# 3 diag
xx, yy, zz = generate_mesh_and_z(sigma1_d, sigma2_d, sigma3_d)

# 2 diag, 1 no diag
xx_1, yy_1, zz_1 = generate_mesh_and_z(sigma1_d, sigma2_d, sigma3)

# 1 diag, 2 no diag
xx_2, yy_2, zz_2 = generate_mesh_and_z(sigma1_d, sigma2, sigma3)
# 3 no diag
xx_3, yy_3, zz_3 = generate_mesh_and_z(sigma1, sigma2, sigma3)

print("X: ", xx)
print("Y: ", yy)
print("Z: ", zz)

X:  [[-2.   -1.92 -1.84 ...  1.84  1.92  2.  ]
 [-2.   -1.92 -1.84 ...  1.84  1.92  2.  ]
 [-2.   -1.92 -1.84 ...  1.84  1.92  2.  ]
 ...
 [-2.   -1.92 -1.84 ...  1.84  1.92  2.  ]
 [-2.   -1.92 -1.84 ...  1.84  1.92  2.  ]
 [-2.   -1.92 -1.84 ...  1.84  1.92  2.  ]]
Y:  [[-2.   -2.   -2.   ... -2.   -2.   -2.  ]
 [-1.92 -1.92 -1.92 ... -1.92 -1.92 -1.92]
 [-1.84 -1.84 -1.84 ... -1.84 -1.84 -1.84]
 ...
 [ 1.84  1.84  1.84 ...  1.84  1.84  1.84]
 [ 1.92  1.92  1.92 ...  1.92  1.92  1.92]
 [ 2.    2.    2.   ...  2.    2.    2.  ]]
Z:  tensor([[1.8926e-02, 2.4447e-02, 3.0913e-02,  ..., 1.1193e-02, 7.8721e-03,
         5.3619e-03],
        [2.4447e-02, 3.1580e-02, 3.9932e-02,  ..., 1.6434e-02, 1.1557e-02,
         7.8721e-03],
        [3.0913e-02, 3.9932e-02, 5.0494e-02,  ..., 2.3367e-02, 1.6434e-02,
         1.1193e-02],
        ...,
        [1.4548e-07, 1.8792e-07, 2.3763e-07,  ..., 9.6119e-03, 6.0114e-03,
         3.6026e-03],
        [6.7493e-08, 8.7185e-08, 1.1024e-07,  ..., 6.0114e-

In [100]:

fig = go.Figure(data=[go.Surface(z=zz_3.numpy(), x=xx_3, y=yy_3)])
fig.update_layout(scene=dict(aspectmode='data'))
fig.update_layout(scene=dict(camera=dict(eye=dict(x=2, y=2, z=1))))

#fig.show()

# Funcion para los sub-bloques

In [101]:
#Get Squeeze Factor for Output values
#Arg:
#  Y (List): A simple list with numbers (float)
def squeze_factor(Y):
  e_f   = 0.0
  max_y = max(Y)
  if max_y > 1:
    e_f = 1/max_y
  else:
    e_f = 1.0
  return e_f

In [102]:
class SubBlock:
    """
    Represents a sub-block in a 2D grid.

    Attributes:
    - vertices (np.ndarray): The vertices of the sub-block.
    - amplitude (int): The amplitude of the sub-block.
    - samples_inside (list): List of samples inside the sub-block.
    - output_values (list): List of output values.
    - index (list): List of index of the original point X

    Methods:
    - add_point(point): Add a point to the sub-block.
    """
    def __init__(self, amplitude=1, ista_layer=None):
        self.amplitude = amplitude
        self.ista_layer = ista_layer
        self._X  = []
        self._index  = []
        self.predicted_output = []
        self.output_values = []

    def add_point(self, point, y):
        self._X.append(point)
        self.output_values.append(y)


def get_sub_block_vertices(grid_size, row, col):
    """
    Get the vertices of a sub-block in a 2D grid.

    Args:
    - grid_size (int): The number of segments per dimension.
    - row (int): The row index of the sub-block.
    - col (int): The column index of the sub-block.

    Returns:
    np.ndarray: The vertices of the sub-block.
    """
    delta = 1.0 / grid_size
    x0 = col * delta
    x1 = (col + 1) * delta
    y0 = row * delta
    y1 = (row + 1) * delta
    return np.array([[x0, y0], [x1, y0], [x0, y1], [x1, y1]])


def locate_samples_in_sub_blocks(x_n, y, t, T):
    """
    Locate points in their respective sub-blocks in a 2D grid.

    Args:
    - x_n (np.ndarray): The normalized points between 0 and 1.
    - y (np.narray) : The output values associated with the samples
    - t (np.ndarray): The integer part of the normalized points.
    - T (int): The number of segments per dimension.

    Returns:
    np.ndarray: Array of SubBlock instances representing the sub-blocks.
    """

    sub_blocks = np.empty((T*T), dtype=object)

    for index in range(T*T):
          sub_blocks[index] = SubBlock()

    for i in range(x_n.shape[0]):
        point = x_n[i]
        location = t[i]
        sub_block = sub_blocks[location[0]*T + location[1]]
        sub_block.add_point(point, y[i])

    return sub_blocks

def generate_list_of_subblock(sub_blocks, l_functions):
  """
    Generate a list for all the sub-blocks with there expected squeze factor

    Arg:
      np.ndarray: Array of SubBlock instances representing the sub-blocks.

    Returns:
    list: List of all sub-block with there block.output_values modified.
  """
  list_sub_blocks = []
  for block in sub_blocks:
    print(f"OUTPUT VALUES: {len(block.output_values) }")
    if(len(block.output_values) != 0):
      block.amplitude = squeze_factor(block.output_values)
      block.ista_layer = ISTALayer(l_functions)
      block.output_values = [value * block.amplitude for value in block.output_values]

      list_sub_blocks.append(block)

  return list_sub_blocks


In [103]:
def data_mapping(X, T):

    norm_x = (X - X.min()) / ( (X.max() - X.min() ) / (T-1))
    t  = norm_x.numpy().astype(int)
    x_n = norm_x - t
    return t, x_n

In [104]:
def predict_on_test_set(X_test, model, T, train_sb):
    t_test, x_n_test = data_mapping(X_test, T)

    sorted_predictions = torch.zeros(len(X_test), dtype=torch.float32)  # Tensor para almacenar las predicciones ordenadas

    for row in range(T):
        for col in range(T):
            sub_block_points = x_n_test[(t_test[:, 0] == row) & (t_test[:, 1] == col)]
            indices = np.where((t_test[:, 0] == row) & (t_test[:, 1] == col))[0]

            if len(sub_block_points) > 0:
                X_sub_block = torch.tensor(sub_block_points, dtype=torch.float32)

                try:
                    current_train_block = train_sb[row * T + col]
                    predictions_sub_block = model.predict(X_sub_block, current_train_block.ista_layer)
                    sorted_predictions[indices] = predictions_sub_block.float() / current_train_block.amplitude
                except IndexError:
                    # No se hace nada para los bloques dummy
                    pass

    return sorted_predictions


# Configuracion y ejecución del modelo

In [105]:
def generate_uniform_sampling(total_points, n_samples=500, min_separation=1):
    """
    Generate uniform sampling indices with a minimum separation criterion.

    Args:
        total_points (int): Total number of data points.
        n_samples (int): Number of samples to generate (default is 500).
        min_separation (int): Minimum separation between selected indices (default is 1).

    Returns:
        list: List of selected indices.

    Example:
        sampled_indices = generate_uniform_sampling(1000, n_samples=200, min_separation=2)
    """

    selected_indexes = []
    while len(selected_indexes) < n_samples:
        random_index = np.random.randint(total_points)
        if all(abs(random_index - existing_index) >= min_separation for existing_index in selected_indexes):
            selected_indexes.append(random_index)
    return selected_indexes

def sample_data(x_values, y_values, z_values, sampled_indices):
    """
    Sample data based on selected indices.

    Args:
        x_values (array-like): X-axis values.
        y_values (array-like): Y-axis values.
        z_values (array-like): Z-axis values.
        sampled_indices (list): List of indices to sample.

    Returns:
        tuple: Tuple containing sampled X (features) and y (labels).

    Example:
        X, y = sample_data(x_values, y_values, z_values, sampled_indices)
    """

    sampled_x = torch.tensor(x_values[sampled_indices], dtype=torch.float32)
    sampled_y = torch.tensor(y_values[sampled_indices], dtype=torch.float32)
    X = torch.stack((sampled_x, sampled_y), dim=1)
    y = z_values[sampled_indices].clone().detach().to(dtype=torch.float32)
    return X, y

def build_model(n_samples, n_features, l_functions, eig_range, mu_range, vector_range):
    """
    Build and configure the SESM (Sparse Evolutionary Structural Modeling) model.

    Args:
    - n_samples (int): Number of training samples.
    - n_features (int): Number of features.
    - l_functions (int): Number of Gaussian functions.

    Returns:
    SESM_Model: An instance of the SESM model.
    """
    gaussian_function = GaussianFunctions(n_features=n_features, n_functions=l_functions, eig_range=eig_range, mu_range=mu_range, vector_range=vector_range)
    model = SESM_Model(
        n_samples=n_samples,
        psi=gaussian_function
    )
    return model

def train_model(model, X, y, model_epochs, ista_epochs, ista_alpha, ista_lambd, dictionary_epochs, dictionary_alpha):
    """
    Train the SESM model using the specified training parameters.

    Args:
    - model (SESM_Model): The SESM model to be trained.
    - X (torch.Tensor): Input features.
    - y (torch.Tensor): Target values.
    - model_epochs (int): Number of epochs for training the overall model.
    - ista_epochs (int): Number of ISTA (Iterative Shrinkage-Thresholding Algorithm) epochs.
    - ista_alpha (float): ISTA alpha parameter.
    - ista_lambd (float): ISTA lambda parameter.
    - dictionary_epochs (int): Number of epochs for updating the dictionary.
    - dictionary_alpha (float): Dictionary update alpha parameter.

    Returns:
    None
    """
    model.fit(
        X=X,
        y=y,
        model_epochs=model_epochs,
        ista_epochs=ista_epochs,
        ista_alpha=ista_alpha,
        ista_lambd=ista_lambd,
        dictionary_epochs=dictionary_epochs,
        dictionary_alpha=dictionary_alpha
    )

def plot_scatter(x_values, y_values, z_values, Z, hypset, fngroup, iteration):
    """
    Plot a scatter plot for the original function and the surrogate model.

    Args:
        x_values (array-like): X-axis values.
        y_values (array-like): Y-axis values.
        z_values (array-like): Z-axis values for the original function.
        Z (array-like): Z-axis values for the surrogate model.
        hypset (int): Hypothesis set or configuration identifier.
        fngroup (int): Function group identifier.
        iteration (int): Iteration number.

    Returns:
        None: Saves the plot as an image file.

    Example:
        plot_scatter(x_values, y_values, z_values, Z, 3, 1 , 1)
    """
    fig = plt.figure(figsize=(13, 13))

    ax1 = fig.add_subplot(221, projection='3d')
    ax1.scatter(x_values, y_values, z_values, c=z_values)
    ax1.set_xlabel('X')
    ax1.set_ylabel('Y')
    ax1.set_zlabel('Z')
    ax1.set_title('Original Function')

    ax2 = fig.add_subplot(222, projection='3d')
    ax2.scatter(x_values, y_values, Z.clone().detach(), c=Z.clone().detach())
    ax2.set_xlabel('X')
    ax2.set_ylabel('Y')
    ax2.set_zlabel('Z')
    ax2.set_title('Surrogate Model')
    ax2.set_zlim(0)

    filename = f"results_{hypset}/plots/{fngroup}.{iteration}.png"
    plt.savefig(filename)

    plt.clf()

def evaluate_model(model, x_values, y_values, z_values, hypset, fngroup, iter, T, list_sub_blocks, debug=True):
    """
    Evaluate the SESM model's performance on a dataset.

    Args:
    - model (SESM_Model): The trained SESM model.
    - x_values (Union[np.ndarray, list]): Input feature values along the x-axis.
    - y_values (Union[np.ndarray, list]): Input feature values along the y-axis.
    - z_values (Union[np.ndarray, list]): Target values.
    - hypset (str): Hypothesis set identifier.
    - fngroup (str): Function group identifier (Dataset).
    - iter (int): Iteration number.
    - debug (bool): Whether to generate and save debugging plots (default is True).

    Returns:
    Tuple[float, float]: Tuple containing time taken for evaluation (in minutes) and
    the Mean Squared Error (MSE) value.
    """
    #----------------PREDICTION------------
    x_tensor = torch.tensor(x_values)
    y_tensor = torch.tensor(y_values)
    X_test   =  torch.stack((x_tensor, y_tensor), dim=1)
    t_test, x_n_test = data_mapping(X_test, T)
    Z = predict_on_test_set(X_test, model, T, list_sub_blocks)

    #----------------- ANALYSIS------------
    time = model.time / 60
    #mse = MeanSquaredError()
    mse = mean_squared_error(Z.clone().detach(), z_values)
    #mse_value = mse.compute()

    if debug:
        plot_scatter(x_values, y_values, z_values, Z, hypset, fngroup, iter)

        df = pd.DataFrame({
            'Loss': model.losses
        })

        df.to_csv(f'results_{hypset}/stats/{iter}.csv', index=False)

        """for loss in model.losses:
          wandb.log({"loss": loss})"""



    return time, mse

def run_experiment(_x, _y, _z, hyperparams, fngroup, iter, debug=True):
    """
    Run a complete SESM experiment, including data sampling, model training, and evaluation.

    Args:
    - _x (np.ndarray): Input feature values along the x-axis.
    - _y (np.ndarray): Input feature values along the y-axis.
    - _z (np.ndarray): Target values.
    - hyperparams (dict): Hyperparameters for the experiment.
    - fngroup (str): Function group identifier.
    - iter (int): Iteration number.
    - debug (bool): Whether to generate and save debugging plots (default is True).

    Returns:
    Tuple[float, float]: Tuple containing time taken for the experiment (in minutes)
    and the Mean Squared Error (MSE) value.
    """
    x_values = _x.ravel()
    y_values = _y.ravel()
    z_values = _z.ravel()

    total_points = len(x_values)

    sampled_indices = generate_uniform_sampling(total_points)
    X, y = sample_data(x_values, y_values, z_values, sampled_indices)

    #SPACE PARTITION
    t, x_n = data_mapping(X, hyperparams["T"])
    sub_blocks = locate_samples_in_sub_blocks(x_n, y, t, hyperparams["T"])
    list_sub_blocks = generate_list_of_subblock(sub_blocks, hyperparams["l_functions"])

    model = build_model(n_samples=hyperparams["n_samples"], n_features=hyperparams["n_features"], l_functions=hyperparams["l_functions"], eig_range=hyperparams["eig_range"], mu_range=hyperparams["mu_range"], vector_range=hyperparams["vector_range"])

    model_epochs = hyperparams["m_epochs"]
    ista_epochs = hyperparams["h_epochs"]
    dictionary_epochs = hyperparams["dict_epochs"]

    ista_alpha = hyperparams["ista_alpha"]
    ista_lambd = hyperparams["ista_lambd"]
    dictionary_alpha = hyperparams["dictionary_alpha"]

    #Aqui lo corremos con el block mayor
#    max_index = max(range(len(list_sub_blocks)), key=lambda i: len(list_sub_blocks[i].output_values))
#    print("max_index",max_index)
#    print("list_sub_blocks[max_index].output_values",len(list_sub_blocks[max_index].output_values))
#    y = torch.tensor(list_sub_blocks[max_index].output_values, dtype=torch.float32)
#    X = torch.tensor(np.array(list_sub_blocks[max_index]._X), dtype=torch.float32)
#    model.ista_layer = list_sub_blocks[max_index].ista_layer
#    train_model(model, X, y, model_epochs, ista_epochs, ista_alpha, ista_lambd, dictionary_epochs, dictionary_alpha)

    for block in list_sub_blocks:
      #if(list_sub_blocks[max_index] !=  block):
        y = torch.tensor(block.output_values, dtype=torch.float32)
        X = torch.tensor(np.array(block._X), dtype=torch.float32)
        model.ista_layer = block.ista_layer
        train_model(model, X, y, model_epochs, ista_epochs, ista_alpha, ista_lambd, dictionary_epochs, dictionary_alpha)
        #block.ista_layer = model.ista_layer

    time, mse_value = evaluate_model(model, x_values, y_values, z_values, hyperparams["hyp_set"],  fngroup, iter, hyperparams["T"], list_sub_blocks, debug)

    return time, mse_value

In [106]:

def plot_stats(directory, num_files):
    """
    Plot statistics for loss values from multiple CSV files.

    Args:
        directory (str): The directory containing CSV files.
        num_files (int): The number of CSV files to process.

    Returns:
        None: Displays an interactive plot and saves an HTML file.

    Note:
        Assumes that each CSV file contains loss values for the same number of epochs.

    """
    fig = px.scatter(title='Loss analysis')
    m_epochs_losses = []

    for i in range(num_files):
        file_path = f"{directory}/stats/{i}.csv"

        df = pd.read_csv(file_path)
        m_epochs_losses.append(df)

    merged_losses = pd.concat(m_epochs_losses, axis=1)

    # Compute mean, std, min, and max for each row
    summary_df = pd.DataFrame({
        'Mean': merged_losses.mean(axis=1),
        'Std': merged_losses.std(axis=1),
        'Min': merged_losses.min(axis=1),
        'Max': merged_losses.max(axis=1)
    })

    summary_df.to_csv(f'{directory}/stats/processed.csv', index=False)

    fig.add_scatter(
        x=summary_df.index,
        y=summary_df['Mean'],
        mode='lines+markers',
        error_y=dict(type='data', array=summary_df['Std']),
        name='Mean'
    )

    fig.add_scatter(
        x=summary_df.index,
        y=summary_df['Max'],
        mode='markers',
        name='Max'
    )

    fig.add_scatter(
        x=summary_df.index,
        y=summary_df['Min'],
        mode='markers',
        name='Min'
    )

    fig.update_layout(
        xaxis_title='m_epochs',
        yaxis_title='Loss',
        legend_title='Legend',
        showlegend=True
    )
    fig.show()
    fig.write_html(f"interactive_plot-{directory}.html")

In [107]:
import csv

def save_results(data, fngroup):
  # Compute Mean and Std for executio
  times = [item[1] for item in data]
  mse_values = [item[2] for item in data]

  average_time = np.mean(times)
  std_time = np.std(times)
  average_mse = np.mean(mse_values)
  std_mse = np.std(mse_values)

  with open(f"results_{fngroup}.csv", mode="w", newline="") as file:
      writer = csv.writer(file)
      writer.writerow(["Repetion", "Time (min)", "MSE"])
      writer.writerows(data)
      writer.writerow(["Mean", average_time, average_mse])
      writer.writerow(["Std", std_time, std_mse])


# Nomenclatura de experimentos

<Set de hiperparámetros>. <Set de datos (conjunto de gaussianas)>.<Número de repetición del experimento>


## Set de Hiperparámetros
|  Hiperparámetro | Exp 1.x.x     | Exp 2.x.x     | Exp 3.x.x     |
|-----------------|---------------|---------------|---------------|
| n_samples       | 50            | 100           | 500           |
| n_features      | 2             | 2             | 2             |
| l_functions     | 20            | 6             | 10            |
| ista_alpha      | 0.06          | 0.0125        | 0.0125        |
| ista_lambd      | 0.005         | 0.001         | 0.001         |
| dictionary_alpha| 0.06          | 0.0125        | 0.0125        |
| m_epochs        | 25            | 500           | 300           |
| dict_epochs     | 800           | 20            | 60            |
| h_epochs        | 1000          | 50            | 100           |


### Set de datos

|     Set      | Exp 1.x.x     | Exp 2.x.x     | Exp 3.x.x     |
|-----------------|---------------|---------------|---------------|
| Gaussianas 1    | Exp 1.1.x     | Exp 2.1.x     | Exp 3.1.x     |
| Gaussianas 2    | Exp 1.2.x     | Exp 2.2.x     | Exp 3.2.x     |
| Gaussianas 3    | Exp 1.3.x     | Exp 2.3.x     | Exp 3.3.x     |
| Gaussianas 4    | Exp 1.4.x     | Exp 2.4.x     | Exp 3.4.x     |


In [108]:
N_iter = 3 #
experiment_3 = {
      "hyp_set": 3,
      "n_samples"	: 500,
      "n_features" : 2,
      "l_functions":  10,
        "eig_range": [1e0, 1e1],
      "mu_range": [-2, 2],
      "vector_range": [1e0, 1e1],
      "ista_alpha"	: 0.05502,
      "ista_lambd"	 : 0.0001007,
      "dictionary_alpha": 0.08928,
      "m_epochs" : 20,
      "dict_epochs": 60,
      "h_epochs": 100,
      "T": 4
}
#1, 2, 3 --> m_epochs


In [109]:
data = []
for i in range(N_iter):
  """wandb.init(
    # Set the project where this run will be logged
    project="SESM-2D-GaussianFunction",
    # We pass a run name (otherwise it’ll be randomly assigned, like sunshine-lollypop-10)
    name=f"12-11-2023-experiments-{i}",
    # Track hyperparameters and run metadata
    config = experiment_3
    )
"""
  time, mse = run_experiment(xx,yy,zz,
                            experiment_3, fngroup=1, iter=i)
  data.append((i, time, mse.item()))
#wandb.finish()
save_results(data=data, fngroup=1)

OUTPUT VALUES: 54
OUTPUT VALUES: 63
OUTPUT VALUES: 53
OUTPUT VALUES: 4
OUTPUT VALUES: 59
OUTPUT VALUES: 64
OUTPUT VALUES: 52
OUTPUT VALUES: 2
OUTPUT VALUES: 49
OUTPUT VALUES: 30
OUTPUT VALUES: 56
OUTPUT VALUES: 4
OUTPUT VALUES: 5
OUTPUT VALUES: 2
OUTPUT VALUES: 3
OUTPUT VALUES: 0
Epoch 1 Loss: 0.04130636900663376

Epoch 2 Loss: 0.031468890607357025

Epoch 3 Loss: 0.021319866180419922

Epoch 4 Loss: 0.014643443748354912

Epoch 5 Loss: 0.011912655085325241

Epoch 6 Loss: 0.01049518957734108

Epoch 7 Loss: 0.00946266483515501

Epoch 8 Loss: 0.008582056500017643

Epoch 9 Loss: 0.007793715689331293

Epoch 10 Loss: 0.007084464654326439

Epoch 11 Loss: 0.006452485453337431

Epoch 12 Loss: 0.005896399263292551

Epoch 13 Loss: 0.005412467289716005

Epoch 14 Loss: 0.004994717426598072

Epoch 15 Loss: 0.004635914694517851

Epoch 16 Loss: 0.004328439477831125

Epoch 17 Loss: 0.004064923617988825

Epoch 18 Loss: 0.0038385987281799316

Epoch 19 Loss: 0.0036434377543628216

Epoch 20 Loss: 0.003474214

<ipython-input-104-b5ebf58b530e>:12: UserWarning:

To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).



OUTPUT VALUES: 53
OUTPUT VALUES: 55
OUTPUT VALUES: 52
OUTPUT VALUES: 3
OUTPUT VALUES: 53
OUTPUT VALUES: 58
OUTPUT VALUES: 48
OUTPUT VALUES: 4
OUTPUT VALUES: 49
OUTPUT VALUES: 54
OUTPUT VALUES: 58
OUTPUT VALUES: 4
OUTPUT VALUES: 2
OUTPUT VALUES: 5
OUTPUT VALUES: 2
OUTPUT VALUES: 0
Epoch 1 Loss: 0.022859986871480942

Epoch 2 Loss: 0.017779115587472916

Epoch 3 Loss: 0.013723643496632576

Epoch 4 Loss: 0.010888049378991127

Epoch 5 Loss: 0.009076337330043316

Epoch 6 Loss: 0.007857379503548145

Epoch 7 Loss: 0.006975545082241297

Epoch 8 Loss: 0.006300522014498711

Epoch 9 Loss: 0.005763054359704256

Epoch 10 Loss: 0.005323383957147598

Epoch 11 Loss: 0.004956813994795084

Epoch 12 Loss: 0.004646875895559788

Epoch 13 Loss: 0.004381900187581778

Epoch 14 Loss: 0.0041531953029334545

Epoch 15 Loss: 0.0039540366269648075

Epoch 16 Loss: 0.0037790988571941853

Epoch 17 Loss: 0.0036240783520042896

Epoch 18 Loss: 0.0034854705445468426

Epoch 19 Loss: 0.0033604044001549482

Epoch 20 Loss: 0.00

<ipython-input-104-b5ebf58b530e>:12: UserWarning:

To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).



OUTPUT VALUES: 65
OUTPUT VALUES: 44
OUTPUT VALUES: 53
OUTPUT VALUES: 3
OUTPUT VALUES: 49
OUTPUT VALUES: 64
OUTPUT VALUES: 55
OUTPUT VALUES: 3
OUTPUT VALUES: 43
OUTPUT VALUES: 60
OUTPUT VALUES: 54
OUTPUT VALUES: 1
OUTPUT VALUES: 2
OUTPUT VALUES: 2
OUTPUT VALUES: 1
OUTPUT VALUES: 1
Epoch 1 Loss: 0.015796223655343056

Epoch 2 Loss: 0.0069342684000730515

Epoch 3 Loss: 0.004499045200645924

Epoch 4 Loss: 0.0035750288516283035

Epoch 5 Loss: 0.003099139081314206

Epoch 6 Loss: 0.0027829045429825783

Epoch 7 Loss: 0.002543906681239605

Epoch 8 Loss: 0.0023553096689283848

Epoch 9 Loss: 0.002205144613981247

Epoch 10 Loss: 0.0020856556948274374

Epoch 11 Loss: 0.0019906959496438503

Epoch 12 Loss: 0.0019151485757902265

Epoch 13 Loss: 0.0018547801300883293

Epoch 14 Loss: 0.0018061540322378278

Epoch 15 Loss: 0.0017665388295426965

Epoch 16 Loss: 0.001733790384605527

Epoch 17 Loss: 0.0017062498955056071

Epoch 18 Loss: 0.0016826431965455413

Epoch 19 Loss: 0.0016620008973404765

Epoch 20 Los

<ipython-input-104-b5ebf58b530e>:12: UserWarning:

To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).



<Figure size 1300x1300 with 0 Axes>

<Figure size 1300x1300 with 0 Axes>

<Figure size 1300x1300 with 0 Axes>

# Correr experimentos

In [ ]:
data_1 = []

for i in range(N_iter):
  """wandb.init(
    # Set the project where this run will be logged
    project="SESM-2D-GaussianFunction",
    # We pass a run name (otherwise it’ll be randomly assigned, like sunshine-lollypop-10)
    name=f"12-11-2023-experiments-{i}",
    # Track hyperparameters and run metadata
    config = experiment_3
    )
"""
  time, mse = run_experiment(xx_1,yy_1,zz_1,
                            experiment_3, fngroup=2, iter=i)
  data_1.append((i, time, mse.item()))

save_results(data=data_1, fngroup=2)

In [ ]:
#Unit test de la inicialización únicamente
stats_dir = f'results_{experiment_3["hyp_set"]}'
plot_stats(stats_dir, N_iter)

In [ ]:
data_2 = []
for i in range(N_iter):
  """  wandb.init(
    # Set the project where this run will be logged
    project="SESM-2D-GaussianFunction",
    # We pass a run name (otherwise it’ll be randomly assigned, like sunshine-lollypop-10)
    name=f"12-11-2023-experiments-{i}",
    # Track hyperparameters and run metadata
    config = experiment_3
    )
"""
  time, mse = run_experiment(xx_2,yy_2,zz_2,
                            experiment_3,fngroup=3, iter=i)
  data_2.append((i, time, mse.item()))

save_results(data=data_2, fngroup=3)

In [43]:
files.download('results_1.csv')
!zip -r results_1.zip results_1/
files.download('results_1.zip')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

  adding: results_1/ (stored 0%)
  adding: results_1/stats/ (stored 0%)
  adding: results_1/plots/ (stored 0%)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
files.download('results_2.csv')
!zip -r results_2.zip results_2/
files.download('results_2.zip')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

  adding: results_2/ (stored 0%)
  adding: results_2/plots/ (stored 0%)
  adding: results_2/stats/ (stored 0%)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [110]:
# files.download('results_3.csv')
!zip -r results_3.zip results_3/
files.download('results_3.zip')

  adding: results_3/ (stored 0%)
  adding: results_3/stats/ (stored 0%)
  adding: results_3/stats/1.csv (deflated 54%)
  adding: results_3/stats/0.csv (deflated 54%)
  adding: results_3/stats/4.csv (deflated 54%)
  adding: results_3/stats/2.csv (deflated 54%)
  adding: results_3/stats/3.csv (deflated 54%)
  adding: results_3/plots/ (stored 0%)
  adding: results_3/plots/.ipynb_checkpoints/ (stored 0%)
  adding: results_3/plots/1.1.png (deflated 4%)
  adding: results_3/plots/1.2.png (deflated 4%)
  adding: results_3/plots/1.0.png (deflated 4%)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>